# Data Processing - Israel Cantons Project

This notebook processes the raw election data using `src.data` modules:
1. Map localities to municipalities using CBS data
2. Aggregate votes from villages to regional councils
3. Apply name fixes and match with GeoJSON
4. Save processed data

**All core logic lives in `src/data/loader.py` and `src/data/processing.py`.**

In [1]:
import sys
sys.path.insert(0, '..')

from src.config import DATA_PROCESSED, KNESSET_IDS, NAME_FIXES
from src.data.loader import (
    load_all_elections, load_cbs_localities,
    load_municipality_geojson, build_locality_to_municipality_mapping,
)
from src.data.processing import process_all_elections

DATA_PROCESSED.mkdir(exist_ok=True)
print("Setup complete")

Setup complete


## 1. Run Full Processing Pipeline

One call handles: load → standardise → aggregate → name-fix → GeoJSON match.

In [2]:
# Full pipeline: load → aggregate → fix names → filter to GeoJSON matches
elections_matched = process_all_elections()

print(f"Processed {len(elections_matched)} elections:")
for knesset, df in elections_matched.items():
    print(f"  Knesset {knesset}: {len(df)} municipalities")

Processed 5 elections:
  Knesset 21: 229 municipalities
  Knesset 22: 229 municipalities
  Knesset 23: 229 municipalities
  Knesset 24: 229 municipalities
  Knesset 25: 229 municipalities


In [3]:
# Inspect a sample
print("Sample columns (Knesset 25):")
print(elections_matched[25].columns.tolist()[:15], "...")
print(f"\nTotal municipalities matched: {len(elections_matched[25])}")

Sample columns (Knesset 25):
['municipality', 'eligible', 'voters', 'invalid', 'valid', 'אמת', 'אצ', 'ב', 'ג', 'ד', 'ום', 'ז', 'זך', 'זנ', 'זץ'] ...

Total municipalities matched: 229


## 2. Inspect Mapping Details

In [4]:
# Locality-to-municipality mapping details
cbs = load_cbs_localities()
locality_to_muni = build_locality_to_municipality_mapping(cbs)
municipalities = set(v for v in locality_to_muni.values() if v is not None)

print(f"CBS localities: {len(cbs)}")
print(f"Mapping entries: {len(locality_to_muni)}")
print(f"Unique municipalities: {len(municipalities)}")
print(f"Name fixes applied: {len(NAME_FIXES)}")

CBS localities: 1483


Mapping entries: 1483
Unique municipalities: 258
Name fixes applied: 41


## 3. GeoJSON Match Check

In [5]:
geo = load_municipality_geojson()
geo_names = set(geo['MUN_HEB'].str.strip())

# Check unmatched (mostly West Bank settlements)
election_names = set(elections_matched[25]['municipality'])
unmatched_geo = geo_names - election_names
print(f"GeoJSON municipalities: {len(geo_names)}")
print(f"Matched: {len(election_names)}")
print(f"In GeoJSON but not matched: {len(unmatched_geo)}")

GeoJSON municipalities: 235
Matched: 229
In GeoJSON but not matched: 6
